**03_model_training.ipynb**

Purpose:
- Define CNN architecture using transfer learning
- Train vehicle classification model
- Evaluate validation performance
- Save trained model for inference

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Vehicle-type-classification")

Mounted at /content/drive


In [2]:
import json
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from src.data_loader import create_generators

In [3]:
with open("artifacts/class_indices.json", "r") as f:
  class_indices = json.load(f)

with open("artifacts/config.json", "r") as f:
    config = json.load(f)


NUM_CLASSES = len(class_indices)

print("Number of classes:", NUM_CLASSES)
print("Class mapping:", class_indices)

Number of classes: 5
Class mapping: {'Bus': 0, 'Car': 1, 'Three Wheeler': 2, 'Truck': 3, 'motorcycle': 4}


In [4]:
train_generator, val_generator = create_generators(config)
print(train_generator.class_indices)

Found 3463 images belonging to 5 classes.
Found 863 images belonging to 5 classes.
{'Bus': 0, 'Car': 1, 'Three Wheeler': 2, 'Truck': 3, 'motorcycle': 4}


In [5]:
from collections import Counter
print(Counter(train_generator.classes))

Counter({np.int32(1): 993, np.int32(0): 740, np.int32(4): 708, np.int32(2): 640, np.int32(3): 382})


In [6]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [7]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
os.makedirs("models", exist_ok=True)

callbacks_stage1 = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

callbacks_stage2 = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(
        "models/vehicle_classifier_best.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]

In [12]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weights = dict(enumerate(class_weights))

In [13]:
history_stage1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks_stage1,
    class_weight=class_weights
)

Epoch 1/10
 11/109 ━━━━━━━━━━━━━━━━━━━━ 15:48 10s/step - accuracy: 0.6016 - loss: 1.1331

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


109/109 ━━━━━━━━━━━━━━━━━━━━ 1549s 14s/step - accuracy: 0.7072 - loss: 0.8620 - val_accuracy: 0.8586 - val_loss: 0.4717
Epoch 2/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 74s 683ms/step - accuracy: 0.8083 - loss: 0.5676 - val_accuracy: 0.8806 - val_loss: 0.3567
Epoch 3/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 75s 687ms/step - accuracy: 0.8530 - loss: 0.4425 - val_accuracy: 0.8876 - val_loss: 0.3234
Epoch 4/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 77s 708ms/step - accuracy: 0.8810 - loss: 0.3849 - val_accuracy: 0.8969 - val_loss: 0.3002
Epoch 5/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 74s 681ms/step - accuracy: 0.8839 - loss: 0.3490 - val_accuracy: 0.8922 - val_loss: 0.2886
Epoch 6/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 75s 689ms/step - accuracy: 0.8958 - loss: 0.3299 - val_accuracy: 0.8992 - val_loss: 0.2773
Epoch 7/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 75s 689ms/step - accuracy: 0.9056 - loss: 0.3045 - val_accuracy: 0.8957 - val_loss: 0.2749
Epoch 8/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 75s 687ms/step - accuracy: 0.9036 - loss: 0.2921 - val

In [14]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_stage2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks_stage2,
    class_weight=class_weights
)

Epoch 1/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 114s 880ms/step - accuracy: 0.8357 - loss: 0.4687 - val_accuracy: 0.9316 - val_loss: 0.2043 - learning_rate: 1.0000e-05
Epoch 2/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 76s 699ms/step - accuracy: 0.8695 - loss: 0.3736 - val_accuracy: 0.9374 - val_loss: 0.2009 - learning_rate: 1.0000e-05
Epoch 3/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 76s 698ms/step - accuracy: 0.8897 - loss: 0.3227 - val_accuracy: 0.9409 - val_loss: 0.1982 - learning_rate: 1.0000e-05
Epoch 4/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 77s 701ms/step - accuracy: 0.8986 - loss: 0.3042 - val_accuracy: 0.9421 - val_loss: 0.1993 - learning_rate: 1.0000e-05
Epoch 5/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 74s 683ms/step - accuracy: 0.9038 - loss: 0.2884 - val_accuracy: 0.9397 - val_loss: 0.2007 - learning_rate: 1.0000e-05
Epoch 6/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 78s 711ms/step - accuracy: 0.9085 - loss: 0.2667 - val_accuracy: 0.9397 - val_loss: 0.1980 - learning_rate: 1.0000e-05
Epoch 7/20
109/109 ━━━━━━━━━━━━━━━━━━━━ 77s 7

In [ ]:
model.save("models/vehicle_mobilenetv2_stage2.keras")

In [ ]:
for i, layer in enumerate(model.layers[0].layers):
    print(i, layer.name, layer.trainable)

0 input_layer False
1 Conv1 False
2 bn_Conv1 False
3 Conv1_relu False
4 expanded_conv_depthwise False
5 expanded_conv_depthwise_BN False
6 expanded_conv_depthwise_relu False
7 expanded_conv_project False
8 expanded_conv_project_BN False
9 block_1_expand False
10 block_1_expand_BN False
11 block_1_expand_relu False
12 block_1_pad False
13 block_1_depthwise False
14 block_1_depthwise_BN False
15 block_1_depthwise_relu False
16 block_1_project False
17 block_1_project_BN False
18 block_2_expand False
19 block_2_expand_BN False
20 block_2_expand_relu False
21 block_2_depthwise False
22 block_2_depthwise_BN False
23 block_2_depthwise_relu False
24 block_2_project False
25 block_2_project_BN False
26 block_2_add False
27 block_3_expand False
28 block_3_expand_BN False
29 block_3_expand_relu False
30 block_3_pad False
31 block_3_depthwise False
32 block_3_depthwise_BN False
33 block_3_depthwise_relu False
34 block_3_project False
35 block_3_project_BN False
36 block_4_expand False
37 block_4_

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Vehicle-type-classification")

import json
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from src.data_loader import create_generators

with open("artifacts/class_indices.json", "r") as f:
  class_indices = json.load(f)

with open("artifacts/config.json", "r") as f:
    config = json.load(f)


NUM_CLASSES = len(class_indices)

print("Number of classes:", NUM_CLASSES)
print("Class mapping:", class_indices)

train_generator, val_generator = create_generators(config)
print(train_generator.class_indices)

from collections import Counter
import numpy as np # Add import for numpy
print(Counter(train_generator.classes))

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

os.makedirs("models", exist_ok=True)

callbacks_stage1 = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

callbacks_stage2 = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(
        "models/vehicle_classifier_best.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]

from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weights = dict(enumerate(class_weights))

history_stage1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks_stage1
)

# Unfreeze the base model
base_model.trainable = True

# Freeze bottom layers (keep generic features)
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_stage2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks_stage2
)

model.save("models/vehicle_mobilenetv2_stage2.keras")

for i, layer in enumerate(model.layers[0].layers):
    print(i, layer.name, layer.trainable)